# Fine-tuning QLoRA — Llama 3.1 8B (Actividad 2 de la tesis)

Este notebook afina **Llama 3.1 8B Instruct** con **QLoRA** sobre el dataset
generado en `backend/scripts/generate_finetune_dataset.py` (planes de
nutrición ya validados por los guardrails clínicos del proyecto).

**Por qué este modelo y no el que usa el RAG en producción:** el backend usa
Llama 3.3 70B vía Together.ai porque es el único modelo de la familia Llama
que Together ofrece en modo serverless — pero el 70B no cabe en la GPU
gratuita de Colab (T4, 16GB VRAM). Por eso el fine-tuning real (esta
Actividad) se hace sobre **Llama 3.1 8B**, descargado directo de Hugging
Face, que sí cabe en 4-bit en una T4. Son etapas distintas a propósito: RAG
primero con un modelo real que funcione (70B), fine-tuning después con el
modelo que de verdad se va a entrenar (8B).

**Antes de correr este notebook:**
1. En Colab: `Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (T4)`.
2. Necesitas acceso aprobado a `meta-llama/Llama-3.1-8B-Instruct` en Hugging
   Face (es un modelo con licencia — pide acceso en su página si no lo tienes,
   normalmente se aprueba en minutos) y un token de HF con permiso de lectura.
3. Sube `plans_train.jsonl` y `plans_val.jsonl` (los que generó el script del
   backend) a este entorno de Colab — ver la celda de "Subir el dataset" abajo.


## 1. Instalar dependencias

In [ ]:
!pip install -q -U transformers accelerate peft bitsandbytes trl datasets huggingface_hub


## 2. Iniciar sesión en Hugging Face

Pide tu token en https://huggingface.co/settings/tokens (basta con permiso de
lectura) y que tu cuenta ya tenga acceso aprobado a
`meta-llama/Llama-3.1-8B-Instruct`.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()


## 3. Subir el dataset

Sube aquí los dos archivos que generó
`backend/scripts/generate_finetune_dataset.py`:
- `plans_train.jsonl`
- `plans_val.jsonl`

(Panel izquierdo → ícono de carpeta → arrastra los archivos, o usa la celda
de abajo para subirlos con un diálogo.)

In [ ]:
from google.colab import files

print("Sube plans_train.jsonl y plans_val.jsonl (puedes seleccionar ambos a la vez):")
uploaded = files.upload()


## 4. Cargar el dataset y aplicar el chat template de Llama 3.1

In [ ]:
import json
from datasets import Dataset
from transformers import AutoTokenizer

BASE_MODEL = "meta-llama/Llama-3.1-8B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.pad_token or tokenizer.eos_token

def load_jsonl(path):
    examples = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                examples.append(json.loads(line))
    return examples

def to_text(example):
    # apply_chat_template arma el formato exacto de turnos system/user/assistant
    # que espera Llama 3.1 — así el modelo afinado ve EXACTAMENTE el mismo
    # formato que va a usar en inferencia real.
    return {"text": tokenizer.apply_chat_template(example["messages"], tokenize=False)}

train_raw = load_jsonl("plans_train.jsonl")
val_raw = load_jsonl("plans_val.jsonl")
print(f"Ejemplos de entrenamiento: {len(train_raw)}  ·  Ejemplos de validación: {len(val_raw)}")

train_dataset = Dataset.from_list(train_raw).map(to_text, remove_columns=["messages"])
val_dataset = Dataset.from_list(val_raw).map(to_text, remove_columns=["messages"]) if val_raw else None

print(train_dataset[0]["text"][:800])


**Nota sobre el tamaño del dataset:** si `len(train_raw)` sale muy chico (ej.
la corrida de prueba con `--count 10` del script), el modelo apenas va a
aprender algo — es normal y esperado para probar que el pipeline completo
funciona de punta a punta. Antes de la entrega final, corre el script de
generación con un `--count` más grande (ej. 150-300) para un fine-tuning que
de verdad se note en la comparación de la Actividad 5.

## 5. Cargar el modelo base en 4-bit (QLoRA)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
)
model.config.use_cache = False


## 6. Configurar LoRA

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    # Proyecciones estándar de attention + MLP en arquitectura Llama.
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


## 7. Entrenar con SFTTrainer

Configuración pensada para una GPU T4 gratuita de Colab (16GB VRAM) — batch
chico + acumulación de gradiente para simular un batch más grande sin
quedarse sin memoria. Si de todos modos truena por memoria, baja
`per_device_train_batch_size` a 1 (ya está en 1) o `max_seq_length`.

In [ ]:
from trl import SFTConfig, SFTTrainer

sft_config = SFTConfig(
    output_dir="./llama31-8b-nutricion-qlora",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=5,
    eval_strategy="epoch" if val_dataset is not None else "no",
    save_strategy="epoch",
    bf16=True,
    max_seq_length=2048,
    dataset_text_field="text",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
)

trainer.train()


## 8. Guardar el adaptador LoRA

Esto guarda SOLO el adaptador (unos cuantos MB), no el modelo completo — para
usarlo después hace falta el modelo base (`meta-llama/Llama-3.1-8B-Instruct`)
+ este adaptador encima.

In [ ]:
ADAPTER_DIR = "./llama31-8b-nutricion-qlora-adapter"
trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"Adaptador guardado en {ADAPTER_DIR}")

# Opcional: respaldarlo a tu Google Drive para no perderlo cuando se cierre
# este entorno de Colab (los entornos de Colab son temporales).
from google.colab import drive
drive.mount("/content/drive")
import shutil
shutil.copytree(ADAPTER_DIR, "/content/drive/MyDrive/llama31-8b-nutricion-qlora-adapter", dirs_exist_ok=True)
print("Copiado a Google Drive.")


## 9. Comparación ANTES/DESPUÉS (para la Actividad 5 de la tesis)

Carga el modelo base **sin** el adaptador y **con** el adaptador, y corre el
mismo prompt de validación en los dos — esto es exactamente la comparación
"antes/después" que pide la Actividad 5 (pruebas y validación) contra el
MISMO modelo base (8B), sin mezclarlo con el 70B que usa el RAG en
producción (ese es un punto de comparación aparte, ya corriendo en el
backend).

In [ ]:
# Reutiliza el primer ejemplo de validación como prompt de prueba. Si no hay
# ejemplos de validación (dataset de prueba muy chico), usa uno de entrenamiento.
sample = (val_raw or train_raw)[0]
system_msg, user_msg = sample["messages"][0], sample["messages"][1]
test_messages = [system_msg, user_msg]
prompt_text = tokenizer.apply_chat_template(test_messages, tokenize=False, add_generation_prompt=True)

inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)

def generate(active_model, label):
    with torch.no_grad():
        output = active_model.generate(**inputs, max_new_tokens=600, do_sample=False)
    text = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    print(f"\n===== {label} =====\n{text}\n")
    return text

print("Generando con el adaptador afinado (modelo actual en memoria)...")
after_text = generate(model, "DESPUÉS del fine-tuning (LoRA aplicado)")

print("Cargando el modelo base SIN adaptador para comparar...")
base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb_config, device_map="auto")
before_text = generate(base_model, "ANTES del fine-tuning (modelo base, sin LoRA)")


**Qué revisar en la comparación:**
- ¿El plan "después" sigue mejor las reglas no negociables del prompt (sin
  alergias, porciones con cantidad y modo de preparación, respeta ayuno
  intermitente, etc.)?
- ¿El JSON del "después" es más consistente/parseable que el de "antes"?
- Corre ambos textos por `run_guardrails()` (ya lo tienes en
  `backend/app/guardrails/engine.py`) para una comparación objetiva, no solo
  lectura a ojo — esa es la evidencia que vale para la Actividad 5.

**Siguiente paso real (no en este notebook):** una vez que el dataset grande
(150-300 ejemplos) esté listo y este entrenamiento corra bien, el adaptador
resultante se puede servir desde un backend propio (ej. con `vllm` o
`text-generation-inference`) para comparar costo/latencia/calidad contra
Together.ai — eso es trabajo de infraestructura adicional, fuera de este
notebook.